# Stage 3: TTA 與 T4 Specialist 融合

讀取 Stage 1/2 artifacts，加入 evidence quality specialist 與 head/middle/tail TTA 融合。

預設 Drive root：`/content/drive/MyDrive/VeriPromiseESG2026`。


## 0. Colab 執行環境與路徑檢查

請先執行下一個 setup cell。它會安裝套件、掛載 Google Drive、檢查 GPU、建立輸出資料夾，並驗證必要輸入檔是否存在。


In [ ]:
# Colab 啟動區塊：安裝套件、掛載 Drive、檢查 GPU、驗證必要輸入檔。
!pip -q install transformers accelerate scikit-learn pandas numpy tqdm openpyxl

from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

BASE_DIR = "/content/drive/MyDrive/VeriPromiseESG2026"
DATA_DIR = f"{BASE_DIR}/data"
OUTPUT_ROOT = f"{BASE_DIR}/outputs"
STAGE_NAME = "stage3_tta_t4_specialist_blend"
STAGE_DISPLAY_NAME = "Stage 3 - TTA 與 T4 Specialist 融合"
OUTPUT_DIR = f"{OUTPUT_ROOT}/{STAGE_NAME}"
OUT_DIR = OUTPUT_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)

REQUIRED_RELATIVE_PATHS = [
    "data/vpesg4k_train_1000.json",
    "data/vpesg4k_valdata_1000.json",
    "data/vpesg4k_testdata_2000.json",
    "outputs/stage1_macbert_multitask_baseline/stage1_oof_val_test_probs.pkl",
    "outputs/stage2_ckip_taskwise_ensemble/stage2_ckip_oof_val_test_probs.pkl",
    "outputs/stage2_ckip_taskwise_ensemble/stage2_best_val_config.json",
]

def stage_log(label, value):
    print(f"[{STAGE_NAME}] {label}: {value}")

def require_files(relative_paths):
    missing = []
    for rel in relative_paths:
        path = f"{BASE_DIR}/{rel}"
        if not os.path.exists(path):
            missing.append(path)
    if missing:
        msg = "缺少必要輸入檔。請先執行前一個 Stage，或將資料放到 Google Drive：\n" + "\n".join(missing)
        raise FileNotFoundError(msg)

require_files(REQUIRED_RELATIVE_PATHS)
stage_log("Stage", STAGE_DISPLAY_NAME)
stage_log("BASE_DIR", BASE_DIR)
stage_log("DATA_DIR", DATA_DIR)
stage_log("OUTPUT_DIR", OUTPUT_DIR)

# 完整訓練建議使用 Colab A100 或同級 GPU。
try:
    gpu_names = !nvidia-smi --query-gpu=name --format=csv,noheader
    gpu_names = list(gpu_names)
    stage_log("GPU", gpu_names)
    if not any("A100" in str(name) for name in gpu_names):
        stage_log("WARNING", "建議使用 A100；非 A100 runtime 可能需要更長時間。")
except Exception as exc:
    stage_log("WARNING", f"無法取得 GPU 資訊，請確認 Colab runtime 已啟用 GPU。{exc}")


## 輸入輸出契約

Stage 顯示名稱：`Stage 3 - TTA 與 T4 Specialist 融合`

Google Drive 輸出資料夾：

```text
/content/drive/MyDrive/VeriPromiseESG2026/outputs/stage3_tta_t4_specialist_blend/
```

必要輸入檔：

- `data/vpesg4k_train_1000.json`
- `data/vpesg4k_valdata_1000.json`
- `data/vpesg4k_testdata_2000.json`
- `outputs/stage1_macbert_multitask_baseline/stage1_oof_val_test_probs.pkl`
- `outputs/stage2_ckip_taskwise_ensemble/stage2_ckip_oof_val_test_probs.pkl`
- `outputs/stage2_ckip_taskwise_ensemble/stage2_best_val_config.json`

主要輸出檔：

- `stage3_t4_specialist_probs.pkl`
- `stage3_3view_probs.pkl`
- `stage3_best_config.json`
- `stage3_best_val_test2000_submission.csv`
- `stage3_conservative_test2000_submission.csv`

若必要輸入不存在，第一個 setup cell 會直接列出缺少的路徑並停止。


## Stage 主要流程

本節針對 evidence_quality 與多視角推論做補強，輸出 Stage 4 會讀取的 TTA probability artifact。


## 1. 路徑與參數設定

In [ ]:

import os
import json
import math
import pickle
import random
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix

from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup

BASE_DIR = "/content/drive/MyDrive/VeriPromiseESG2026"
DATA_DIR = f"{BASE_DIR}/data"

TRAIN_PATH = f"{DATA_DIR}/vpesg4k_train_1000.json"
VAL_PATH   = f"{DATA_DIR}/vpesg4k_valdata_1000.json"
TEST_PATH  = f"{DATA_DIR}/vpesg4k_testdata_2000.json"

V30_DIR = f"{BASE_DIR}/outputs/stage1_macbert_multitask_baseline"
MACBERT_PROB_PATH = f"{V30_DIR}/stage1_oof_val_test_probs.pkl"

V31_DIR = f"{BASE_DIR}/outputs/stage2_ckip_taskwise_ensemble"
CKIP_PROB_PATH = f"{V31_DIR}/stage2_ckip_oof_val_test_probs.pkl"
V31_CONFIG_PATH = f"{V31_DIR}/stage2_best_val_config.json"

OUT_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
os.makedirs(OUT_DIR, exist_ok=True)

TASKS = [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
]

LABELS = {
    "promise_status": ["No", "Yes"],
    "verification_timeline": ["N/A", "already", "within_2_years", "between_2_and_5_years", "more_than_5_years"],
    "evidence_status": ["N/A", "No", "Yes"],
    "evidence_quality": ["N/A", "Clear", "Not Clear", "Misleading"],
}

T4_LABELS = ["Clear", "Not Clear", "Misleading"]
T4_LABEL2ID = {lab: i for i, lab in enumerate(T4_LABELS)}
T4_ID2LABEL = {i: lab for lab, i in T4_LABEL2ID.items()}

LABEL2ID = {t: {lab: i for i, lab in enumerate(labs)} for t, labs in LABELS.items()}
ID2LABEL = {t: {i: lab for i, lab in enumerate(labs)} for t, labs in LABELS.items()}

MODEL_NAME = "hfl/chinese-macbert-base"
SEED = 44
N_SPLITS = 5
MAX_LEN = 384
EPOCHS = 5
BATCH_SIZE = 8
GRAD_ACCUM = 2
LR = 2e-5
WEIGHT_DECAY = 0.01
PATIENCE = 2
NUM_WORKERS = 0

# T4 specialist 使用 mild class weights，但上限不高，避免又推歪
USE_T4_CLASS_WEIGHT = True
T4_CLASS_WEIGHT_MAX = 4.0

SKIP_TRAIN_IF_PROBS_EXIST = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
print("OUT_DIR:", OUT_DIR)


In [ ]:

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)


## 2. 讀取資料、stage1 / stage2 機率與 stage2 設定

In [ ]:

def load_json_df(path):
    with open(path, "r", encoding="utf-8") as f:
        return pd.DataFrame(json.load(f))

train_df = load_json_df(TRAIN_PATH)
val_df = load_json_df(VAL_PATH)
test_df = load_json_df(TEST_PATH)

for df in [train_df, val_df]:
    for t in TASKS:
        df[t] = df[t].fillna("N/A").astype(str).replace({"": "N/A", "nan": "N/A"})

with open(MACBERT_PROB_PATH, "rb") as f:
    mac_obj = pickle.load(f)
mac_val_probs = mac_obj["val_probs"]
mac_test_probs = mac_obj["test_probs"]

with open(CKIP_PROB_PATH, "rb") as f:
    ckip_obj = pickle.load(f)
ckip_val_probs = ckip_obj["val_probs"]
ckip_test_probs = ckip_obj["test_probs"]

with open(V31_CONFIG_PATH, "r", encoding="utf-8") as f:
    v31_obj = json.load(f)
V31_BEST_CONFIG = v31_obj["best_config"]

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)
print("v31 best config:")
print(json.dumps(V31_BEST_CONFIG, ensure_ascii=False, indent=2))

print("\nT4 train distribution:")
print(train_df["evidence_quality"].value_counts().to_dict())
print("\nT4 val distribution:")
print(val_df["evidence_quality"].value_counts().to_dict())


## 3. 評分、後處理與 stage2 base 重建工具

In [ ]:

def apply_hierarchy_to_pred_df(pred_df, force_quality_non_na_when_evidence_yes=False, probs=None):
    pred_df = pred_df.copy()

    no_mask = pred_df["promise_status"] == "No"
    pred_df.loc[no_mask, "verification_timeline"] = "N/A"
    pred_df.loc[no_mask, "evidence_status"] = "N/A"
    pred_df.loc[no_mask, "evidence_quality"] = "N/A"

    not_evidence_yes = pred_df["evidence_status"] != "Yes"
    pred_df.loc[not_evidence_yes, "evidence_quality"] = "N/A"

    if force_quality_non_na_when_evidence_yes and probs is not None:
        mask = (pred_df["evidence_status"] == "Yes") & (pred_df["evidence_quality"] == "N/A")
        if mask.any():
            q_probs = probs["evidence_quality"][mask.values].copy()
            q_probs[:, LABEL2ID["evidence_quality"]["N/A"]] = -1
            ids = q_probs.argmax(axis=1)
            pred_df.loc[mask, "evidence_quality"] = [ID2LABEL["evidence_quality"][int(i)] for i in ids]

    return pred_df

def f1_yes(y_true, y_pred):
    y_true_bin = (pd.Series(y_true).astype(str).values == "Yes").astype(int)
    y_pred_bin = (pd.Series(y_pred).astype(str).values == "Yes").astype(int)
    return f1_score(y_true_bin, y_pred_bin, pos_label=1, average="binary", zero_division=0)

def macro_f1_task(y_true, y_pred, task):
    return f1_score(
        y_true,
        y_pred,
        labels=LABELS[task],
        average="macro",
        zero_division=0,
    )

def calc_metrics(y_true_df, pred_df):
    out = {}
    out["promise_status_f1_yes"] = f1_yes(y_true_df["promise_status"], pred_df["promise_status"])
    out["verification_timeline_macro_f1"] = macro_f1_task(y_true_df["verification_timeline"], pred_df["verification_timeline"], "verification_timeline")
    out["evidence_status_f1_yes"] = f1_yes(y_true_df["evidence_status"], pred_df["evidence_status"])
    out["evidence_quality_macro_f1"] = macro_f1_task(y_true_df["evidence_quality"], pred_df["evidence_quality"], "evidence_quality")
    out["weighted_score"] = (
        0.20 * out["promise_status_f1_yes"]
        + 0.15 * out["verification_timeline_macro_f1"]
        + 0.30 * out["evidence_status_f1_yes"]
        + 0.35 * out["evidence_quality_macro_f1"]
    )
    return out

def print_metrics(m, title="metrics"):
    print("\n" + "="*80)
    print(title)
    print("="*80)
    for k, v in m.items():
        print(f"{k:36s}: {v:.6f}")

def adjust_probs(probs, multipliers):
    out = {}
    for t in TASKS:
        p = probs[t].copy()
        if t in multipliers:
            mult = np.array([multipliers[t].get(lab, 1.0) for lab in LABELS[t]], dtype=np.float32)
            p = p * mult.reshape(1, -1)
            p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
        out[t] = p
    return out

def combine_probs(mac_probs, ckip_probs, weights):
    out = {}
    for t in TASKS:
        w = weights.get(t, 0.5)
        out[t] = w * mac_probs[t] + (1 - w) * ckip_probs[t]
    return out

def probs_to_pred_df(df, probs, config=None):
    if config is None:
        config = {}

    multipliers = config.get("multipliers", {})
    force_q_non_na = config.get("force_quality_non_na_when_evidence_yes", False)
    t1_threshold = config.get("t1_threshold", None)
    t3_threshold = config.get("t3_threshold", None)

    probs2 = adjust_probs(probs, multipliers)

    pred = pd.DataFrame()
    pred["id"] = df["id"].values

    if t1_threshold is not None:
        yes_id = LABEL2ID["promise_status"]["Yes"]
        pred["promise_status"] = np.where(probs2["promise_status"][:, yes_id] >= t1_threshold, "Yes", "No")
    else:
        ids = probs2["promise_status"].argmax(axis=1)
        pred["promise_status"] = [ID2LABEL["promise_status"][int(i)] for i in ids]

    ids = probs2["verification_timeline"].argmax(axis=1)
    pred["verification_timeline"] = [ID2LABEL["verification_timeline"][int(i)] for i in ids]

    if t3_threshold is not None:
        yes_id = LABEL2ID["evidence_status"]["Yes"]
        pred_es = []
        for row_probs in probs2["evidence_status"]:
            if row_probs[yes_id] >= t3_threshold:
                pred_es.append("Yes")
            else:
                na_id = LABEL2ID["evidence_status"]["N/A"]
                no_id = LABEL2ID["evidence_status"]["No"]
                pred_es.append("N/A" if row_probs[na_id] >= row_probs[no_id] else "No")
        pred["evidence_status"] = pred_es
    else:
        ids = probs2["evidence_status"].argmax(axis=1)
        pred["evidence_status"] = [ID2LABEL["evidence_status"][int(i)] for i in ids]

    ids = probs2["evidence_quality"].argmax(axis=1)
    pred["evidence_quality"] = [ID2LABEL["evidence_quality"][int(i)] for i in ids]

    pred = apply_hierarchy_to_pred_df(
        pred,
        force_quality_non_na_when_evidence_yes=force_q_non_na,
        probs=probs2,
    )
    return pred

def get_v31_base_probs(split):
    weights = V31_BEST_CONFIG["weights"]
    if split == "val":
        return combine_probs(mac_val_probs, ckip_val_probs, weights)
    if split == "test":
        return combine_probs(mac_test_probs, ckip_test_probs, weights)
    raise ValueError(split)

def get_v31_base_pred(split):
    base_probs = get_v31_base_probs(split)
    df = val_df if split == "val" else test_df
    post_cfg = {k: v for k, v in V31_BEST_CONFIG.items() if k != "weights"}
    return probs_to_pred_df(df, base_probs, post_cfg), base_probs

v31_val_pred, v31_val_probs = get_v31_base_pred("val")
v31_metrics = calc_metrics(val_df, v31_val_pred)
print_metrics(v31_metrics, "v31 base reconstructed")

for t in TASKS:
    print(t, v31_val_pred[t].value_counts().to_dict())


## 4. 建立 T4 specialist 訓練資料

In [ ]:

# T4 specialist 只學 evidence_status = Yes 的 quality
t4_train_df = train_df[
    (train_df["evidence_status"] == "Yes") &
    (train_df["evidence_quality"].isin(T4_LABELS))
].copy().reset_index(drop=True)

t4_val_true_df = val_df[
    (val_df["evidence_status"] == "Yes") &
    (val_df["evidence_quality"].isin(T4_LABELS))
].copy().reset_index(drop=True)

print("t4_train:", t4_train_df.shape)
print(t4_train_df["evidence_quality"].value_counts().to_dict())
print("t4_val_true:", t4_val_true_df.shape)
print(t4_val_true_df["evidence_quality"].value_counts().to_dict())

display(t4_train_df.head())


## 5. T4 Specialist Dataset / Model / Loss

In [ ]:

class T4Dataset(Dataset):
    def __init__(self, df, tokenizer, max_len=384, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            str(row["data"]),
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors=None,
        )

        item = {
            "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
        }
        if "token_type_ids" in enc:
            item["token_type_ids"] = torch.tensor(enc["token_type_ids"], dtype=torch.long)

        if self.has_labels:
            item["labels"] = torch.tensor(T4_LABEL2ID[row["evidence_quality"]], dtype=torch.long)
        return item

class T4QualityClassifier(nn.Module):
    def __init__(self, model_name, n_labels=3, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden * 2, n_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids

        out = self.encoder(**kwargs)
        last_hidden = out.last_hidden_state
        cls_vec = last_hidden[:, 0, :]
        mask = attention_mask.unsqueeze(-1).float()
        mean_vec = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        pooled = torch.cat([cls_vec, mean_vec], dim=-1)
        return self.classifier(self.dropout(pooled))

def make_t4_loss(fold_train_df):
    if not USE_T4_CLASS_WEIGHT:
        return nn.CrossEntropyLoss()

    counts = fold_train_df["evidence_quality"].value_counts().to_dict()
    total = len(fold_train_df)
    n_classes = len(T4_LABELS)
    arr = []
    for lab in T4_LABELS:
        c = max(counts.get(lab, 0), 1)
        arr.append(total / (n_classes * c))
    w = torch.tensor(arr, dtype=torch.float)
    w = torch.clamp(w, max=T4_CLASS_WEIGHT_MAX)
    print("class weights:", dict(zip(T4_LABELS, w.tolist())))
    return nn.CrossEntropyLoss(weight=w.to(DEVICE))

def batch_to_device(batch):
    out = {}
    for k, v in batch.items():
        out[k] = v.to(DEVICE)
    return out

@torch.no_grad()
def predict_t4_probs(model, df, tokenizer, batch_size=16):
    model.eval()
    ds = T4Dataset(df, tokenizer, max_len=MAX_LEN, has_labels=False)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    probs_list = []

    for batch in tqdm(dl, desc="predict_t4", leave=False):
        batch = batch_to_device(batch)
        logits = model(**batch)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        probs_list.append(probs)

    return np.concatenate(probs_list, axis=0)


## 6. T4 specialist 5-fold 訓練

In [ ]:

def make_specialist_folds(df, n_splits=5, seed=42):
    # Misleading 通常只有 1 筆，無法正常 StratifiedKFold。
    # 做法：少於 n_splits 的 rare class 永遠放在 train，不放進 fold valid。
    counts = df["evidence_quality"].value_counts()
    rare_labels = set(counts[counts < n_splits].index.tolist())

    rare_idx = df.index[df["evidence_quality"].isin(rare_labels)].to_numpy()
    base_idx = df.index[~df["evidence_quality"].isin(rare_labels)].to_numpy()

    base_y = df.loc[base_idx, "evidence_quality"].values

    print("rare_labels always in train:", rare_labels)
    print("rare_idx:", rare_idx.tolist())

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = []
    for tr_base, va_base in skf.split(base_idx, base_y):
        tr_idx = np.concatenate([base_idx[tr_base], rare_idx])
        va_idx = base_idx[va_base]
        folds.append((tr_idx, va_idx))
    return folds

def train_t4_one_fold(fold, fold_train_df, fold_valid_df, tokenizer):
    print(f"\n========== T4 Fold {fold} ==========")
    print("train:", fold_train_df.shape, fold_train_df["evidence_quality"].value_counts().to_dict())
    print("valid:", fold_valid_df.shape, fold_valid_df["evidence_quality"].value_counts().to_dict())

    model = T4QualityClassifier(MODEL_NAME, n_labels=len(T4_LABELS), dropout=0.2).to(DEVICE)

    train_ds = T4Dataset(fold_train_df, tokenizer, max_len=MAX_LEN, has_labels=True)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = math.ceil(len(train_dl) / GRAD_ACCUM) * EPOCHS
    warmup_steps = int(total_steps * 0.1)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    loss_fn = make_t4_loss(fold_train_df)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

    best_score = -1
    best_path = f"{OUT_DIR}/t4_specialist_fold{fold}_best.pt"
    bad_epochs = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss = 0.0

        pbar = tqdm(train_dl, desc=f"t4 fold{fold} epoch{epoch}", leave=False)
        for step, batch in enumerate(pbar, start=1):
            batch = batch_to_device(batch)
            labels = batch.pop("labels")

            with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
                logits = model(**batch)
                loss = loss_fn(logits, labels) / GRAD_ACCUM

            scaler.scale(loss).backward()
            total_loss += loss.item() * GRAD_ACCUM

            if step % GRAD_ACCUM == 0 or step == len(train_dl):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            pbar.set_postfix(loss=total_loss / step)

        valid_probs = predict_t4_probs(model, fold_valid_df, tokenizer, batch_size=16)
        pred_ids = valid_probs.argmax(axis=1)
        y_pred = [T4_ID2LABEL[int(i)] for i in pred_ids]
        y_true = fold_valid_df["evidence_quality"].tolist()
        score = f1_score(y_true, y_pred, labels=T4_LABELS, average="macro", zero_division=0)

        print(f"T4 fold {fold} epoch {epoch} macro-F1: {score:.6f}")
        print("pred dist:", Counter(y_pred))

        if score > best_score:
            best_score = score
            bad_epochs = 0
            torch.save(model.state_dict(), best_path)
            print(f"saved best T4 fold{fold}: {best_score:.6f}")
        else:
            bad_epochs += 1
            print("bad_epochs:", bad_epochs)
            if bad_epochs >= PATIENCE:
                print("early stopping")
                break

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    return model, best_score, best_path


In [ ]:

T4_PROB_PATH = f"{OUT_DIR}/stage3_t4_specialist_probs.pkl"

if SKIP_TRAIN_IF_PROBS_EXIST and os.path.exists(T4_PROB_PATH):
    print("Loading existing T4 specialist probs:", T4_PROB_PATH)
    with open(T4_PROB_PATH, "rb") as f:
        t4_obj = pickle.load(f)
    t4_val_probs = t4_obj["val_probs"]
    t4_test_probs = t4_obj["test_probs"]
    t4_fold_scores = t4_obj.get("fold_scores", [])
    t4_fold_paths = t4_obj.get("fold_paths", [])
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    folds = make_specialist_folds(t4_train_df, n_splits=N_SPLITS, seed=SEED)

    t4_val_probs_sum = np.zeros((len(val_df), len(T4_LABELS)), dtype=np.float32)
    t4_test_probs_sum = np.zeros((len(test_df), len(T4_LABELS)), dtype=np.float32)
    t4_fold_scores = []
    t4_fold_paths = []

    for fold, (tr_idx, va_idx) in enumerate(folds, start=1):
        fold_train_df = t4_train_df.loc[tr_idx].reset_index(drop=True)
        fold_valid_df = t4_train_df.loc[va_idx].reset_index(drop=True)

        model, best_score, best_path = train_t4_one_fold(fold, fold_train_df, fold_valid_df, tokenizer)
        t4_fold_scores.append(best_score)
        t4_fold_paths.append(best_path)

        val_probs = predict_t4_probs(model, val_df, tokenizer, batch_size=16)
        t4_val_probs_sum += val_probs / N_SPLITS

        test_probs = predict_t4_probs(model, test_df, tokenizer, batch_size=16)
        t4_test_probs_sum += test_probs / N_SPLITS

        del model
        torch.cuda.empty_cache()

    t4_val_probs = t4_val_probs_sum
    t4_test_probs = t4_test_probs_sum

    t4_obj = {
        "model_name": MODEL_NAME,
        "t4_labels": T4_LABELS,
        "train_path": TRAIN_PATH,
        "val_path": VAL_PATH,
        "test_path": TEST_PATH,
        "fold_scores": t4_fold_scores,
        "fold_paths": t4_fold_paths,
        "val_probs": t4_val_probs,
        "test_probs": t4_test_probs,
        "val_ids": val_df["id"].tolist(),
        "test_ids": test_df["id"].tolist(),
    }
    with open(T4_PROB_PATH, "wb") as f:
        pickle.dump(t4_obj, f)

    print("saved:", T4_PROB_PATH)

print("T4 fold scores:", t4_fold_scores)
if len(t4_fold_scores):
    print("mean:", np.mean(t4_fold_scores))
print("val probs:", t4_val_probs.shape)
print("test probs:", t4_test_probs.shape)


## 7. 將 T4 specialist 接回 stage2 base 預測

In [ ]:

def t4_probs_to_labels(spec_probs, spec_multipliers=None):
    p = spec_probs.copy()
    if spec_multipliers:
        mult = np.array([spec_multipliers.get(lab, 1.0) for lab in T4_LABELS], dtype=np.float32)
        p = p * mult.reshape(1, -1)
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
    ids = p.argmax(axis=1)
    return [T4_ID2LABEL[int(i)] for i in ids], p

def get_base_quality_non_na_probs(base_probs, base_config):
    # 取得 v31 base 的 adjusted quality probs，並轉成 Clear/Not Clear/Misleading 三類
    multipliers = base_config.get("multipliers", {})
    adjusted = adjust_probs(base_probs, multipliers)
    q = adjusted["evidence_quality"]
    non_na_ids = [
        LABEL2ID["evidence_quality"]["Clear"],
        LABEL2ID["evidence_quality"]["Not Clear"],
        LABEL2ID["evidence_quality"]["Misleading"],
    ]
    q3 = q[:, non_na_ids].copy()
    q3 = q3 / np.clip(q3.sum(axis=1, keepdims=True), 1e-12, None)
    return q3

def compose_with_t4_specialist(
    df,
    base_pred,
    base_probs,
    spec_probs,
    mode="replace",
    alpha=1.0,
    spec_multipliers=None,
):
    out = base_pred.copy()

    # 只有 base evidence_status = Yes 的樣本才使用 quality specialist
    mask = out["evidence_status"] == "Yes"

    if mode == "replace":
        spec_labels, _ = t4_probs_to_labels(spec_probs, spec_multipliers=spec_multipliers)
        out.loc[mask, "evidence_quality"] = np.array(spec_labels)[mask.values]

    elif mode == "blend":
        base_q3 = get_base_quality_non_na_probs(base_probs, V31_BEST_CONFIG)
        _, spec_q3 = t4_probs_to_labels(spec_probs, spec_multipliers=spec_multipliers)
        blend = (1 - alpha) * base_q3 + alpha * spec_q3
        blend = blend / np.clip(blend.sum(axis=1, keepdims=True), 1e-12, None)
        ids = blend.argmax(axis=1)
        labels = [T4_ID2LABEL[int(i)] for i in ids]
        out.loc[mask, "evidence_quality"] = np.array(labels)[mask.values]

    else:
        raise ValueError(mode)

    out.loc[~mask, "evidence_quality"] = "N/A"
    out = apply_hierarchy_to_pred_df(out)
    return out

# baseline v31
print_metrics(calc_metrics(val_df, v31_val_pred), "stage2 base")

replace_pred = compose_with_t4_specialist(
    val_df,
    v31_val_pred,
    v31_val_probs,
    t4_val_probs,
    mode="replace",
    spec_multipliers=None,
)
replace_metrics = calc_metrics(val_df, replace_pred)
print_metrics(replace_metrics, "stage3A T4 specialist replace")

for t in TASKS:
    print(t, replace_pred[t].value_counts().to_dict())


## 8. 搜尋 T4 specialist blend / calibration

In [ ]:

def eval_t4_config(alpha, spec_multipliers):
    pred = compose_with_t4_specialist(
        val_df,
        v31_val_pred,
        v31_val_probs,
        t4_val_probs,
        mode="blend",
        alpha=alpha,
        spec_multipliers=spec_multipliers,
    )
    metrics = calc_metrics(val_df, pred)
    return metrics["weighted_score"], metrics, pred

alpha_values = [round(x, 2) for x in np.arange(0.0, 1.01, 0.05)]
mult_values = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0]

best = None
rows = []

for alpha in alpha_values:
    for clear_m in mult_values:
        for notclear_m in mult_values:
            for misleading_m in [0.5, 1.0, 2.0, 4.0]:
                spec_mult = {
                    "Clear": clear_m,
                    "Not Clear": notclear_m,
                    "Misleading": misleading_m,
                }
                score, metrics, pred = eval_t4_config(alpha, spec_mult)
                row = {
                    "alpha": alpha,
                    "clear_m": clear_m,
                    "notclear_m": notclear_m,
                    "misleading_m": misleading_m,
                    **metrics,
                    "pred_Clear": int((pred["evidence_quality"] == "Clear").sum()),
                    "pred_Not_Clear": int((pred["evidence_quality"] == "Not Clear").sum()),
                    "pred_Misleading": int((pred["evidence_quality"] == "Misleading").sum()),
                    "pred_NA": int((pred["evidence_quality"] == "N/A").sum()),
                }
                rows.append(row)
                if best is None or score > best[0]:
                    best = (score, metrics, pred, alpha, spec_mult)

search_df = pd.DataFrame(rows).sort_values("weighted_score", ascending=False)
search_df.to_csv(f"{OUT_DIR}/stage3_t4_blend_search.csv", index=False, encoding="utf-8-sig")

best_score, best_metrics, best_pred, best_alpha, best_spec_mult = best
print_metrics(best_metrics, "BEST stage3A T4 blend val1000")
print("best_alpha:", best_alpha)
print("best_spec_mult:", best_spec_mult)
for t in TASKS:
    print(t, best_pred[t].value_counts().to_dict())

display(search_df.head(20))


## 9. 儲存 val1000 錯誤分析與設定

In [ ]:

def save_val_outputs(pred, metrics, tag):
    val_full = pd.concat(
        [
            val_df[["id", "data", "company", "page_number"]].reset_index(drop=True),
            val_df[TASKS].add_prefix("true_").reset_index(drop=True),
            pred[TASKS].add_prefix("pred_").reset_index(drop=True),
        ],
        axis=1,
    )
    for t in TASKS:
        val_full[f"err_{t}"] = val_full[f"true_{t}"] != val_full[f"pred_{t}"]
    val_full["any_error"] = val_full[[f"err_{t}" for t in TASKS]].any(axis=1)

    val_full.to_csv(f"{OUT_DIR}/{tag}_val1000_predictions.csv", index=False, encoding="utf-8-sig")
    val_full.to_excel(f"{OUT_DIR}/{tag}_val1000_error_analysis.xlsx", index=False)

    with pd.ExcelWriter(f"{OUT_DIR}/{tag}_val1000_confusion.xlsx", engine="openpyxl") as writer:
        for t in TASKS:
            cm = confusion_matrix(val_df[t], pred[t], labels=LABELS[t])
            cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in LABELS[t]], columns=[f"pred_{x}" for x in LABELS[t]])
            cm_df.to_excel(writer, sheet_name=t[:31])

    return val_full

best_val_full = save_val_outputs(best_pred, best_metrics, "stage3_t4_best_val")
replace_val_full = save_val_outputs(replace_pred, replace_metrics, "stage3_t4_replace")

config_obj = {
    "base": "stage2_best_val",
    "mode": "blend",
    "alpha": best_alpha,
    "spec_multipliers": best_spec_mult,
    "v31_metrics": v31_metrics,
    "replace_metrics": replace_metrics,
    "best_metrics": best_metrics,
    "t4_fold_scores": t4_fold_scores,
    "model_name": MODEL_NAME,
}

with open(f"{OUT_DIR}/stage3_t4_best_config.json", "w", encoding="utf-8") as f:
    json.dump(config_obj, f, ensure_ascii=False, indent=2)

print("saved config and val outputs to:", OUT_DIR)
display(best_val_full["any_error"].value_counts())
display(best_val_full.head())


## 10. 產生 test2000 submission

In [ ]:

def compact_text(s):
    return re.sub(r"\s+", "", str(s))

train_map = {}
for _, row in train_df.iterrows():
    train_map[compact_text(row["data"])] = {t: row[t] for t in TASKS}

def make_test_submission(tag, mode="blend", alpha=None, spec_multipliers=None):
    base_test_pred, base_test_probs = get_v31_base_pred("test")

    if mode == "base":
        pred = base_test_pred.copy()
    elif mode == "replace":
        pred = compose_with_t4_specialist(
            test_df,
            base_test_pred,
            base_test_probs,
            t4_test_probs,
            mode="replace",
            spec_multipliers=spec_multipliers,
        )
    elif mode == "blend":
        pred = compose_with_t4_specialist(
            test_df,
            base_test_pred,
            base_test_probs,
            t4_test_probs,
            mode="blend",
            alpha=alpha,
            spec_multipliers=spec_multipliers,
        )
    else:
        raise ValueError(mode)

    override_count = 0
    for i, row in test_df.iterrows():
        key = compact_text(row["data"])
        if key in train_map:
            for t in TASKS:
                pred.loc[i, t] = train_map[key][t]
            override_count += 1

    pred = apply_hierarchy_to_pred_df(pred)

    sub = pred[["id"] + TASKS].copy()
    path = f"{OUT_DIR}/{tag}_test2000_submission.csv"
    sub.to_csv(path, index=False, encoding="utf-8-sig")

    print("\n", tag, "saved:", path)
    print("override_count:", override_count)
    for t in TASKS:
        print(t, sub[t].value_counts().to_dict())

    return sub, path

base_sub, base_path = make_test_submission("stage3_t4_base_stage2", mode="base")
replace_sub, replace_path = make_test_submission("stage3_t4_replace", mode="replace", spec_multipliers=None)
best_sub, best_path = make_test_submission(
    "stage3_t4_best_blend",
    mode="blend",
    alpha=best_alpha,
    spec_multipliers=best_spec_mult,
)

# conservative：降低 specialist 影響，避免只在 val 過度校正
conservative_alpha = min(best_alpha, 0.50)
conservative_sub, conservative_path = make_test_submission(
    "stage3_t4_conservative_blend",
    mode="blend",
    alpha=conservative_alpha,
    spec_multipliers=best_spec_mult,
)

print("\nSubmission paths:")
print(base_path)
print(replace_path)
print(best_path)
print(conservative_path)


## 11. 報告摘要

In [ ]:

report = []
report.append("# stage_3A T4 quality specialist report")
report.append("")
report.append("## stage2 base metrics")
for k, v in v31_metrics.items():
    report.append(f"- {k}: {v:.6f}")
report.append("")
report.append("## T4 specialist replace metrics")
for k, v in replace_metrics.items():
    report.append(f"- {k}: {v:.6f}")
report.append("")
report.append("## Best T4 blend metrics")
for k, v in best_metrics.items():
    report.append(f"- {k}: {v:.6f}")
report.append("")
report.append("## Best config")
report.append("```json")
report.append(json.dumps(config_obj, ensure_ascii=False, indent=2))
report.append("```")
report.append("")
report.append("## Test distribution: stage3_t4_best_blend")
for t in TASKS:
    report.append(f"### {t}")
    for lab, cnt in best_sub[t].value_counts().to_dict().items():
        report.append(f"- {lab}: {cnt}")

report_md = "\n".join(report)

with open(f"{OUT_DIR}/stage3_t4_report.md", "w", encoding="utf-8") as f:
    f.write(report_md)

print(report_md[:5000])
print("saved:", f"{OUT_DIR}/stage3_t4_report.md")



## 12. 判讀方式

請看最後輸出的：

1. `stage2 base metrics`
2. `T4 specialist replace metrics`
3. `Best T4 blend metrics`
4. `Test distribution`

如果 `Best T4 blend metrics` 比 stage2 的 `0.6902` 高，而且 test 分布沒有過度偏斜，就可以把 `stage3_t4_best_blend_test2000_submission.csv` 當成下一個候選。

如果沒有提升，代表單靠 T4 specialist 不夠，下一步應做：

- T4 minority augmentation
- T2 specialist
- 3-view TTA


## Stage 3B head/middle/tail TTA

This section uses public Stage paths and artifact names under `/content/drive/MyDrive/VeriPromiseESG2026`.


## 1. 路徑與設定

In [ ]:

import os
import json
import math
import pickle
import random
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import f1_score, confusion_matrix

from transformers import AutoTokenizer, AutoModel

BASE_DIR = "/content/drive/MyDrive/VeriPromiseESG2026"
DATA_DIR = f"{BASE_DIR}/data"

TRAIN_PATH = f"{DATA_DIR}/vpesg4k_train_1000.json"
VAL_PATH   = f"{DATA_DIR}/vpesg4k_valdata_1000.json"
TEST_PATH  = f"{DATA_DIR}/vpesg4k_testdata_2000.json"

V30_DIR = f"{BASE_DIR}/outputs/stage1_macbert_multitask_baseline"
MACBERT_PROB_PATH = f"{V30_DIR}/stage1_oof_val_test_probs.pkl"

V31_DIR = f"{BASE_DIR}/outputs/stage2_ckip_taskwise_ensemble"
CKIP_PROB_PATH = f"{V31_DIR}/stage2_ckip_oof_val_test_probs.pkl"
V31_CONFIG_PATH = f"{V31_DIR}/stage2_best_val_config.json"

# v32_00 只作為比較，不是必要
V32_00_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
V32_00_CONFIG_PATH = f"{V32_00_DIR}/stage3_t4_best_config.json"

OUT_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
os.makedirs(OUT_DIR, exist_ok=True)

VIEW_PROB_PATH = f"{OUT_DIR}/stage3_3view_probs.pkl"

TASKS = [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
]

LABELS = {
    "promise_status": ["No", "Yes"],
    "verification_timeline": ["N/A", "already", "within_2_years", "between_2_and_5_years", "more_than_5_years"],
    "evidence_status": ["N/A", "No", "Yes"],
    "evidence_quality": ["N/A", "Clear", "Not Clear", "Misleading"],
}

LABEL2ID = {t: {lab: i for i, lab in enumerate(labs)} for t, labs in LABELS.items()}
ID2LABEL = {t: {i: lab for i, lab in enumerate(labs)} for t, labs in LABELS.items()}

MAX_LEN = 384
BATCH_SIZE = 16
NUM_WORKERS = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
print("OUT_DIR:", OUT_DIR)
print("VIEW_PROB_PATH:", VIEW_PROB_PATH)


In [ ]:

def seed_everything(seed=45):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(45)


## 2. 讀取資料、probs、fold paths、config

In [ ]:

def load_json_df(path):
    with open(path, "r", encoding="utf-8") as f:
        return pd.DataFrame(json.load(f))

train_df = load_json_df(TRAIN_PATH)
val_df = load_json_df(VAL_PATH)
test_df = load_json_df(TEST_PATH)

for df in [train_df, val_df]:
    for t in TASKS:
        df[t] = df[t].fillna("N/A").astype(str).replace({"": "N/A", "nan": "N/A"})

with open(MACBERT_PROB_PATH, "rb") as f:
    mac_obj = pickle.load(f)
mac_val_probs_head = mac_obj["val_probs"]
mac_test_probs_head = mac_obj["test_probs"]
mac_fold_paths = mac_obj["fold_paths"]
mac_model_name = mac_obj.get("model_name", "hfl/chinese-macbert-base")

with open(CKIP_PROB_PATH, "rb") as f:
    ckip_obj = pickle.load(f)
ckip_val_probs_head = ckip_obj["val_probs"]
ckip_test_probs_head = ckip_obj["test_probs"]
ckip_fold_paths = ckip_obj["fold_paths"]
ckip_model_name = ckip_obj.get("model_name", "ckiplab/bert-base-chinese")

with open(V31_CONFIG_PATH, "r", encoding="utf-8") as f:
    v31_obj = json.load(f)
V31_BEST_CONFIG = v31_obj["best_config"]

if os.path.exists(V32_00_CONFIG_PATH):
    with open(V32_00_CONFIG_PATH, "r", encoding="utf-8") as f:
        V32_00_CONFIG_OBJ = json.load(f)
else:
    V32_00_CONFIG_OBJ = None

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)
print("mac_model:", mac_model_name)
print("ckip_model:", ckip_model_name)
print("mac_fold_paths:", mac_fold_paths)
print("ckip_fold_paths:", ckip_fold_paths)
print("v31 best config:")
print(json.dumps(V31_BEST_CONFIG, ensure_ascii=False, indent=2))


## 3. 評分與後處理工具

In [ ]:

def apply_hierarchy_to_pred_df(pred_df, force_quality_non_na_when_evidence_yes=False, probs=None):
    pred_df = pred_df.copy()

    no_mask = pred_df["promise_status"] == "No"
    pred_df.loc[no_mask, "verification_timeline"] = "N/A"
    pred_df.loc[no_mask, "evidence_status"] = "N/A"
    pred_df.loc[no_mask, "evidence_quality"] = "N/A"

    not_evidence_yes = pred_df["evidence_status"] != "Yes"
    pred_df.loc[not_evidence_yes, "evidence_quality"] = "N/A"

    if force_quality_non_na_when_evidence_yes and probs is not None:
        mask = (pred_df["evidence_status"] == "Yes") & (pred_df["evidence_quality"] == "N/A")
        if mask.any():
            q_probs = probs["evidence_quality"][mask.values].copy()
            q_probs[:, LABEL2ID["evidence_quality"]["N/A"]] = -1
            ids = q_probs.argmax(axis=1)
            pred_df.loc[mask, "evidence_quality"] = [ID2LABEL["evidence_quality"][int(i)] for i in ids]

    return pred_df

def f1_yes(y_true, y_pred):
    y_true_bin = (pd.Series(y_true).astype(str).values == "Yes").astype(int)
    y_pred_bin = (pd.Series(y_pred).astype(str).values == "Yes").astype(int)
    return f1_score(y_true_bin, y_pred_bin, pos_label=1, average="binary", zero_division=0)

def macro_f1_task(y_true, y_pred, task):
    return f1_score(
        y_true,
        y_pred,
        labels=LABELS[task],
        average="macro",
        zero_division=0,
    )

def calc_metrics(y_true_df, pred_df):
    out = {}
    out["promise_status_f1_yes"] = f1_yes(y_true_df["promise_status"], pred_df["promise_status"])
    out["verification_timeline_macro_f1"] = macro_f1_task(y_true_df["verification_timeline"], pred_df["verification_timeline"], "verification_timeline")
    out["evidence_status_f1_yes"] = f1_yes(y_true_df["evidence_status"], pred_df["evidence_status"])
    out["evidence_quality_macro_f1"] = macro_f1_task(y_true_df["evidence_quality"], pred_df["evidence_quality"], "evidence_quality")
    out["weighted_score"] = (
        0.20 * out["promise_status_f1_yes"]
        + 0.15 * out["verification_timeline_macro_f1"]
        + 0.30 * out["evidence_status_f1_yes"]
        + 0.35 * out["evidence_quality_macro_f1"]
    )
    return out

def print_metrics(m, title="metrics"):
    print("\n" + "="*80)
    print(title)
    print("="*80)
    for k, v in m.items():
        print(f"{k:36s}: {v:.6f}")

def adjust_probs(probs, multipliers):
    out = {}
    for t in TASKS:
        p = probs[t].copy()
        if t in multipliers:
            mult = np.array([multipliers[t].get(lab, 1.0) for lab in LABELS[t]], dtype=np.float32)
            p = p * mult.reshape(1, -1)
            p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
        out[t] = p
    return out

def probs_to_pred_df(df, probs, config=None):
    if config is None:
        config = {}

    multipliers = config.get("multipliers", {})
    force_q_non_na = config.get("force_quality_non_na_when_evidence_yes", False)
    t1_threshold = config.get("t1_threshold", None)
    t3_threshold = config.get("t3_threshold", None)

    probs2 = adjust_probs(probs, multipliers)

    pred = pd.DataFrame()
    pred["id"] = df["id"].values

    if t1_threshold is not None:
        yes_id = LABEL2ID["promise_status"]["Yes"]
        pred["promise_status"] = np.where(probs2["promise_status"][:, yes_id] >= t1_threshold, "Yes", "No")
    else:
        ids = probs2["promise_status"].argmax(axis=1)
        pred["promise_status"] = [ID2LABEL["promise_status"][int(i)] for i in ids]

    ids = probs2["verification_timeline"].argmax(axis=1)
    pred["verification_timeline"] = [ID2LABEL["verification_timeline"][int(i)] for i in ids]

    if t3_threshold is not None:
        yes_id = LABEL2ID["evidence_status"]["Yes"]
        pred_es = []
        for row_probs in probs2["evidence_status"]:
            if row_probs[yes_id] >= t3_threshold:
                pred_es.append("Yes")
            else:
                na_id = LABEL2ID["evidence_status"]["N/A"]
                no_id = LABEL2ID["evidence_status"]["No"]
                pred_es.append("N/A" if row_probs[na_id] >= row_probs[no_id] else "No")
        pred["evidence_status"] = pred_es
    else:
        ids = probs2["evidence_status"].argmax(axis=1)
        pred["evidence_status"] = [ID2LABEL["evidence_status"][int(i)] for i in ids]

    ids = probs2["evidence_quality"].argmax(axis=1)
    pred["evidence_quality"] = [ID2LABEL["evidence_quality"][int(i)] for i in ids]

    pred = apply_hierarchy_to_pred_df(
        pred,
        force_quality_non_na_when_evidence_yes=force_q_non_na,
        probs=probs2,
    )
    return pred


## 4. Model 與 view dataset

In [ ]:

class MultiTaskClassifier(nn.Module):
    def __init__(self, model_name, label_dims, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        pooled_dim = hidden * 2
        self.dropouts = nn.ModuleDict({t: nn.Dropout(dropout) for t in TASKS})
        self.classifiers = nn.ModuleDict({t: nn.Linear(pooled_dim, label_dims[t]) for t in TASKS})

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids

        out = self.encoder(**kwargs)
        last_hidden = out.last_hidden_state
        cls_vec = last_hidden[:, 0, :]
        mask = attention_mask.unsqueeze(-1).float()
        mean_vec = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        pooled = torch.cat([cls_vec, mean_vec], dim=-1)

        return {t: self.classifiers[t](self.dropouts[t](pooled)) for t in TASKS}

class ViewDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=384, view="head"):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.view = view

        # 避免使用 build_inputs_with_special_tokens / prepare_for_model，
        # 因為不同 tokenizer 版本或 wrapper 可能沒有這些 method。
        self.cls_id = getattr(tokenizer, "cls_token_id", None)
        self.sep_id = getattr(tokenizer, "sep_token_id", None)
        self.pad_id = getattr(tokenizer, "pad_token_id", None)

        if self.cls_id is None:
            self.cls_id = tokenizer.convert_tokens_to_ids("[CLS]")
        if self.sep_id is None:
            self.sep_id = tokenizer.convert_tokens_to_ids("[SEP]")
        if self.pad_id is None:
            self.pad_id = 0

        # BERT 類模型：[CLS] content [SEP]
        self.special_n = 2

    def __len__(self):
        return len(self.df)

    def select_tokens(self, token_ids):
        max_content_len = self.max_len - self.special_n

        if len(token_ids) <= max_content_len:
            return token_ids

        if self.view == "head":
            return token_ids[:max_content_len]

        if self.view == "tail":
            return token_ids[-max_content_len:]

        if self.view == "middle":
            start = max((len(token_ids) - max_content_len) // 2, 0)
            return token_ids[start:start + max_content_len]

        raise ValueError(f"unknown view: {self.view}")

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]["data"])

        # 只用最基本的 encode，避免 tokenizer helper method 版本不相容
        token_ids = self.tokenizer.encode(text, add_special_tokens=False)
        token_ids = self.select_tokens(token_ids)

        input_ids = [self.cls_id] + token_ids + [self.sep_id]
        attention_mask = [1] * len(input_ids)
        token_type_ids = [0] * len(input_ids)

        pad_len = self.max_len - len(input_ids)
        if pad_len > 0:
            input_ids += [self.pad_id] * pad_len
            attention_mask += [0] * pad_len
            token_type_ids += [0] * pad_len
        else:
            input_ids = input_ids[:self.max_len]
            attention_mask = attention_mask[:self.max_len]
            token_type_ids = token_type_ids[:self.max_len]

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "token_type_ids": torch.tensor(token_type_ids, dtype=torch.long),
        }

def batch_to_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}

@torch.no_grad()
def predict_view_probs_for_model(model, df, tokenizer, view, batch_size=16):
    model.eval()
    ds = ViewDataset(df, tokenizer, max_len=MAX_LEN, view=view)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    all_probs = {t: [] for t in TASKS}

    for batch in tqdm(dl, desc=f"predict {view}", leave=False):
        batch = batch_to_device(batch)
        logits = model(**batch)
        for t in TASKS:
            all_probs[t].append(torch.softmax(logits[t], dim=-1).cpu().numpy())

    return {t: np.concatenate(all_probs[t], axis=0) for t in TASKS}


## 5. 建立 / 載入 3-view probs 快取

In [ ]:

def average_probs(list_of_probs):
    if not list_of_probs:
        raise ValueError("No probability arrays were produced; check checkpoint loading first.")
    avg = {}
    for t in TASKS:
        avg[t] = np.mean([p[t] for p in list_of_probs], axis=0).astype(np.float32)
    return avg

def validate_checkpoint_paths(stem_name, fold_paths):
    """Fail early if a fold checkpoint is missing or likely corrupted."""
    bad = []
    for i, ckpt_path in enumerate(fold_paths, start=1):
        path = Path(str(ckpt_path))
        if not path.exists():
            bad.append(f"fold {i}: missing file: {path}")
            continue
        size = path.stat().st_size
        if size <= 0:
            bad.append(f"fold {i}: empty checkpoint ({size} bytes): {path}")
        elif size < 1024:
            bad.append(f"fold {i}: suspiciously small checkpoint ({size} bytes): {path}")
    if bad:
        joined = "\n".join(bad)
        raise FileNotFoundError(
            f"Invalid {stem_name} fold checkpoint(s).\n"
            f"{joined}\n\n"
            "Fix: rerun the previous Stage that trained this stem, then rerun this cell. "
            "For CKIP, rerun Stage2 from the CKIP training section and regenerate "
            "stage2_ckip_oof_val_test_probs.pkl. If a zero-byte *.pt file exists in Drive, "
            "delete it before rerunning."
        )

def load_checkpoint_state(stem_name, fold_idx, ckpt_path):
    path = Path(str(ckpt_path))
    try:
        return torch.load(path, map_location=DEVICE)
    except (EOFError, RuntimeError, pickle.UnpicklingError) as exc:
        raise RuntimeError(
            f"Failed to load {stem_name} fold {fold_idx} checkpoint: {path}\n"
            f"File size: {path.stat().st_size if path.exists() else 'missing'} bytes\n"
            "This usually means the checkpoint is incomplete/corrupted on Google Drive. "
            "Delete the bad checkpoint and rerun the previous Stage that produced it."
        ) from exc

def load_view_prob_cache(path):
    try:
        with open(path, "rb") as f:
            return pickle.load(f)
    except (EOFError, pickle.UnpicklingError) as exc:
        raise RuntimeError(
            f"Failed to load cached view probabilities: {path}\n"
            "The cache file is incomplete/corrupted. Delete it and rerun this cell to recompute."
        ) from exc

def compute_stem_view_probs(stem_name, model_name, fold_paths, val_head_probs, test_head_probs):
    validate_checkpoint_paths(stem_name, fold_paths)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    result = {
        "val": {"head": val_head_probs},
        "test": {"head": test_head_probs},
    }

    for view in ["middle", "tail"]:
        print(f"\n=== Computing {stem_name} {view} ===")
        val_fold_probs = []
        test_fold_probs = []

        for i, ckpt_path in enumerate(fold_paths, start=1):
            print(f"Loading {stem_name} fold {i}: {ckpt_path}")

            model = MultiTaskClassifier(
                model_name,
                label_dims={t: len(LABELS[t]) for t in TASKS},
                dropout=0.2,
            ).to(DEVICE)

            state = load_checkpoint_state(stem_name, i, ckpt_path)
            model.load_state_dict(state)

            val_probs = predict_view_probs_for_model(model, val_df, tokenizer, view=view, batch_size=BATCH_SIZE)
            test_probs = predict_view_probs_for_model(model, test_df, tokenizer, view=view, batch_size=BATCH_SIZE)

            val_fold_probs.append(val_probs)
            test_fold_probs.append(test_probs)

            del model
            torch.cuda.empty_cache()

        result["val"][view] = average_probs(val_fold_probs)
        result["test"][view] = average_probs(test_fold_probs)

    return result

# Validate all checkpoints before doing expensive view inference.
validate_checkpoint_paths("mac", mac_fold_paths)
validate_checkpoint_paths("ckip", ckip_fold_paths)

if os.path.exists(VIEW_PROB_PATH):
    print("Loading existing view probs:", VIEW_PROB_PATH)
    view_obj = load_view_prob_cache(VIEW_PROB_PATH)
else:
    view_obj = {}
    view_obj["mac"] = compute_stem_view_probs(
        stem_name="mac",
        model_name=mac_model_name,
        fold_paths=mac_fold_paths,
        val_head_probs=mac_val_probs_head,
        test_head_probs=mac_test_probs_head,
    )
    view_obj["ckip"] = compute_stem_view_probs(
        stem_name="ckip",
        model_name=ckip_model_name,
        fold_paths=ckip_fold_paths,
        val_head_probs=ckip_val_probs_head,
        test_head_probs=ckip_test_probs_head,
    )
    view_obj["meta"] = {
        "mac_model_name": mac_model_name,
        "ckip_model_name": ckip_model_name,
        "mac_fold_paths": mac_fold_paths,
        "ckip_fold_paths": ckip_fold_paths,
        "views": ["head", "middle", "tail"],
        "val_ids": val_df["id"].tolist(),
        "test_ids": test_df["id"].tolist(),
    }
    tmp_path = VIEW_PROB_PATH + ".tmp"
    with open(tmp_path, "wb") as f:
        pickle.dump(view_obj, f)
    os.replace(tmp_path, VIEW_PROB_PATH)
    print("saved:", VIEW_PROB_PATH)

for stem in ["mac", "ckip"]:
    for split in ["val", "test"]:
        for view in ["head", "middle", "tail"]:
            print(stem, split, view, {t: view_obj[stem][split][view][t].shape for t in TASKS})


## 6. View TTA 組合與 baseline 檢查

In [ ]:

VIEW_PRESETS = {
    "head": {"head": 1.0, "middle": 0.0, "tail": 0.0},
    "middle": {"head": 0.0, "middle": 1.0, "tail": 0.0},
    "tail": {"head": 0.0, "middle": 0.0, "tail": 1.0},

    "head_mid": {"head": 0.70, "middle": 0.30, "tail": 0.0},
    "head_tail": {"head": 0.70, "middle": 0.0, "tail": 0.30},
    "mid_tail": {"head": 0.0, "middle": 0.50, "tail": 0.50},

    "even": {"head": 0.34, "middle": 0.33, "tail": 0.33},
    "head_mid_tail": {"head": 0.50, "middle": 0.25, "tail": 0.25},
    "mid_more": {"head": 0.30, "middle": 0.50, "tail": 0.20},
    "tail_more": {"head": 0.30, "middle": 0.20, "tail": 0.50},
    "head_tail_more": {"head": 0.50, "middle": 0.10, "tail": 0.40},
    "head_mid_more": {"head": 0.50, "middle": 0.40, "tail": 0.10},
}

def mix_views(stem_view_probs, split, task, preset_name):
    preset = VIEW_PRESETS[preset_name]
    out = None
    for view, w in preset.items():
        p = stem_view_probs[split][view][task]
        out = w * p if out is None else out + w * p
    return out

def build_tta_probs(split, config):
    weights = config.get("weights", {t: 0.5 for t in TASKS})
    view_presets = config.get("view_presets", {t: "head" for t in TASKS})

    out = {}
    for t in TASKS:
        preset_name = view_presets.get(t, "head")
        mac_p = mix_views(view_obj["mac"], split, t, preset_name)
        ckip_p = mix_views(view_obj["ckip"], split, t, preset_name)

        w_mac = weights.get(t, 0.5)
        p = w_mac * mac_p + (1 - w_mac) * ckip_p
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
        out[t] = p.astype(np.float32)

    return out

def eval_tta_config(config, verbose=False, title=""):
    probs = build_tta_probs("val", config)
    post_cfg = {k: v for k, v in config.items() if k not in ["weights", "view_presets"]}
    pred = probs_to_pred_df(val_df, probs, post_cfg)
    metrics = calc_metrics(val_df, pred)

    if verbose:
        print_metrics(metrics, title or "tta config")
        print("weights:", json.dumps(config.get("weights", {}), ensure_ascii=False, indent=2))
        print("view_presets:", json.dumps(config.get("view_presets", {}), ensure_ascii=False, indent=2))
        print("post_cfg:", json.dumps(post_cfg, ensure_ascii=False, indent=2))
        for t in TASKS:
            print(t, pred[t].value_counts().to_dict())

    return metrics["weighted_score"], metrics, pred

# v31 reconstructed: all head + v31 weights/config
v31_like_config = json.loads(json.dumps(V31_BEST_CONFIG))
v31_like_config["view_presets"] = {t: "head" for t in TASKS}
score, metrics, pred = eval_tta_config(v31_like_config, verbose=True, title="stage2 reconstructed from head views")


## 7. 搜尋 v33 3-view TTA

In [ ]:

def copy_config(cfg):
    return json.loads(json.dumps(cfg))

candidate_starts = []

# start 1: v31 best, all head
cfg = copy_config(v31_like_config)
candidate_starts.append(("stage2_head", cfg))

# start 2: same config, timeline/evidence/quality use balanced views
cfg = copy_config(v31_like_config)
cfg["view_presets"] = {
    "promise_status": "head",
    "verification_timeline": "head_mid_tail",
    "evidence_status": "head_tail",
    "evidence_quality": "head_tail",
}
candidate_starts.append(("semantic_views", cfg))

# start 3: more tail-oriented for T3/T4
cfg = copy_config(v31_like_config)
cfg["view_presets"] = {
    "promise_status": "head",
    "verification_timeline": "mid_more",
    "evidence_status": "tail_more",
    "evidence_quality": "tail_more",
}
candidate_starts.append(("tail_for_evidence", cfg))

best_start = None
for name, cfg in candidate_starts:
    score, metrics, pred = eval_tta_config(cfg, verbose=True, title=f"start: {name}")
    if best_start is None or score > best_start[0]:
        best_start = (score, cfg, metrics)

print("best start:", best_start[0])
print(json.dumps(best_start[1], ensure_ascii=False, indent=2))


In [ ]:

def coordinate_search(base_config, search_space, rounds=5):
    best_cfg = copy_config(base_config)
    best_score, best_metrics, best_pred = eval_tta_config(best_cfg)
    history = []

    for r in range(rounds):
        improved = False
        print(f"\n--- coordinate round {r+1} start score {best_score:.6f} ---")

        for key, values in search_space:
            local_best = (best_score, copy_config(best_cfg), best_metrics)

            for val in values:
                cfg = copy_config(best_cfg)

                if key[0] == "weight":
                    _, task = key
                    cfg.setdefault("weights", {})[task] = val

                elif key[0] == "view":
                    _, task = key
                    cfg.setdefault("view_presets", {})[task] = val

                elif key[0] == "mult":
                    _, task, label = key
                    cfg.setdefault("multipliers", {}).setdefault(task, {})[label] = val

                elif key[0] == "threshold":
                    _, name = key
                    cfg[name] = val

                elif key[0] == "flag":
                    _, name = key
                    cfg[name] = val

                score, metrics, _ = eval_tta_config(cfg)
                if score > local_best[0]:
                    local_best = (score, cfg, metrics)

            if local_best[0] > best_score + 1e-9:
                best_score, best_cfg, best_metrics = local_best
                improved = True
                print("improved", key, "->", best_score)
                print(json.dumps(best_cfg, ensure_ascii=False, indent=2))

        history.append({"round": r+1, "score": best_score, "config": copy_config(best_cfg)})

        if not improved:
            break

    best_score, best_metrics, best_pred = eval_tta_config(best_cfg)
    return best_cfg, best_score, best_metrics, best_pred, history

weight_values = [round(x, 2) for x in np.arange(0.0, 1.01, 0.05)]
threshold_values = [round(x, 2) for x in np.arange(0.30, 0.81, 0.02)]
mult_values = [0.10, 0.20, 0.35, 0.50, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0, 4.0, 6.0]

view_values = list(VIEW_PRESETS.keys())

search_space = []

# 每個任務的 MacBERT/CKIP 權重
for t in TASKS:
    search_space.append((("weight", t), weight_values))

# 每個任務的 view preset
for t in TASKS:
    search_space.append((("view", t), view_values))

# T4 / T2 multipliers
for lab in LABELS["evidence_quality"]:
    search_space.append((("mult", "evidence_quality", lab), mult_values))

for lab in LABELS["verification_timeline"]:
    search_space.append((("mult", "verification_timeline", lab), mult_values))

# T1 / T3 thresholds
search_space.append((("threshold", "t1_threshold"), [None] + threshold_values))
search_space.append((("threshold", "t3_threshold"), [None] + threshold_values))

# optional flag
search_space.append((("flag", "force_quality_non_na_when_evidence_yes"), [False, True]))

best_cfg, best_score, best_metrics, best_pred, history = coordinate_search(
    best_start[1],
    search_space,
    rounds=5,
)

print_metrics(best_metrics, "BEST stage3B 3-view TTA val1000")
print("best config:")
print(json.dumps(best_cfg, ensure_ascii=False, indent=2))
for t in TASKS:
    print("\n", t)
    print(best_pred[t].value_counts().to_dict())


## 8. 儲存 val1000 分析

In [ ]:

def save_val_outputs(pred, metrics, tag):
    val_full = pd.concat(
        [
            val_df[["id", "data", "company", "page_number"]].reset_index(drop=True),
            val_df[TASKS].add_prefix("true_").reset_index(drop=True),
            pred[TASKS].add_prefix("pred_").reset_index(drop=True),
        ],
        axis=1,
    )

    for t in TASKS:
        val_full[f"err_{t}"] = val_full[f"true_{t}"] != val_full[f"pred_{t}"]

    val_full["any_error"] = val_full[[f"err_{t}" for t in TASKS]].any(axis=1)

    val_full.to_csv(f"{OUT_DIR}/{tag}_val1000_predictions.csv", index=False, encoding="utf-8-sig")
    val_full.to_excel(f"{OUT_DIR}/{tag}_val1000_error_analysis.xlsx", index=False)

    with pd.ExcelWriter(f"{OUT_DIR}/{tag}_val1000_confusion.xlsx", engine="openpyxl") as writer:
        for t in TASKS:
            cm = confusion_matrix(val_df[t], pred[t], labels=LABELS[t])
            cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in LABELS[t]], columns=[f"pred_{x}" for x in LABELS[t]])
            cm_df.to_excel(writer, sheet_name=t[:31])

    return val_full

best_val_full = save_val_outputs(best_pred, best_metrics, "stage3_tta_best_val")

with open(f"{OUT_DIR}/stage3_tta_best_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "best_config": best_cfg,
        "best_metrics": best_metrics,
        "history": history,
        "v31_like_metrics": metrics,
    }, f, ensure_ascii=False, indent=2)

print("saved val outputs and config to:", OUT_DIR)
display(best_val_full["any_error"].value_counts())
display(best_val_full.head())


## 9. 產生 best / conservative / balanced submissions

In [ ]:

def compact_text(s):
    return re.sub(r"\s+", "", str(s))

train_map = {}
for _, row in train_df.iterrows():
    train_map[compact_text(row["data"])] = {t: row[t] for t in TASKS}

def make_test_submission(config, tag):
    probs = build_tta_probs("test", config)
    post_cfg = {k: v for k, v in config.items() if k not in ["weights", "view_presets"]}
    pred = probs_to_pred_df(test_df, probs, post_cfg)

    override_count = 0
    for i, row in test_df.iterrows():
        key = compact_text(row["data"])
        if key in train_map:
            for t in TASKS:
                pred.loc[i, t] = train_map[key][t]
            override_count += 1

    pred = apply_hierarchy_to_pred_df(pred)

    sub = pred[["id"] + TASKS].copy()
    path = f"{OUT_DIR}/{tag}_test2000_submission.csv"
    sub.to_csv(path, index=False, encoding="utf-8-sig")

    print("\n", tag, "saved:", path)
    print("override_count:", override_count)
    for t in TASKS:
        print(t, sub[t].value_counts().to_dict())

    return sub, path

best_sub, best_path = make_test_submission(best_cfg, "stage3_tta_best_val")

# conservative：提高 T1/T3 threshold，減少 cascade error
conservative_cfg = copy_config(best_cfg)
if conservative_cfg.get("t1_threshold") is not None:
    conservative_cfg["t1_threshold"] = min(float(conservative_cfg["t1_threshold"]) + 0.04, 0.80)
if conservative_cfg.get("t3_threshold") is not None:
    conservative_cfg["t3_threshold"] = min(float(conservative_cfg["t3_threshold"]) + 0.04, 0.80)
conservative_sub, conservative_path = make_test_submission(conservative_cfg, "stage3_tta_conservative")

# balanced：降低較激進的 within_2_years / Not Clear multiplier
balanced_cfg = copy_config(best_cfg)
balanced_cfg.setdefault("multipliers", {})

if "verification_timeline" in balanced_cfg["multipliers"]:
    if "within_2_years" in balanced_cfg["multipliers"]["verification_timeline"]:
        balanced_cfg["multipliers"]["verification_timeline"]["within_2_years"] = max(
            1.0,
            float(balanced_cfg["multipliers"]["verification_timeline"]["within_2_years"]) * 0.75
        )

if "evidence_quality" in balanced_cfg["multipliers"]:
    if "Not Clear" in balanced_cfg["multipliers"]["evidence_quality"]:
        balanced_cfg["multipliers"]["evidence_quality"]["Not Clear"] = max(
            0.5,
            float(balanced_cfg["multipliers"]["evidence_quality"]["Not Clear"]) * 0.85
        )

balanced_sub, balanced_path = make_test_submission(balanced_cfg, "stage3_tta_balanced")

print("\nSubmission paths:")
print(best_path)
print(conservative_path)
print(balanced_path)


## 10. 報告摘要

In [ ]:

report = []
report.append("# stage_3B 3-view TTA report")
report.append("")
report.append("## Best stage_3B metrics")
for k, v in best_metrics.items():
    report.append(f"- {k}: {v:.6f}")

report.append("")
report.append("## Best config")
report.append("```json")
report.append(json.dumps(best_cfg, ensure_ascii=False, indent=2))
report.append("```")

report.append("")
report.append("## Test distribution: stage3_tta_best_val")
for t in TASKS:
    report.append(f"### {t}")
    for lab, cnt in best_sub[t].value_counts().to_dict().items():
        report.append(f"- {lab}: {cnt}")

report_md = "\n".join(report)

with open(f"{OUT_DIR}/stage3_tta_report.md", "w", encoding="utf-8") as f:
    f.write(report_md)

print(report_md[:5000])
print("saved:", f"{OUT_DIR}/stage3_tta_report.md")


## Stage 3C TTA and T4 blend

This section uses public Stage paths and artifact names under `/content/drive/MyDrive/VeriPromiseESG2026`.


## 1. 路徑與設定

In [ ]:

import os
import json
import pickle
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, confusion_matrix

BASE_DIR = "/content/drive/MyDrive/VeriPromiseESG2026"
DATA_DIR = f"{BASE_DIR}/data"

TRAIN_PATH = f"{DATA_DIR}/vpesg4k_train_1000.json"
VAL_PATH   = f"{DATA_DIR}/vpesg4k_valdata_1000.json"
TEST_PATH  = f"{DATA_DIR}/vpesg4k_testdata_2000.json"

V31_DIR = f"{BASE_DIR}/outputs/stage2_ckip_taskwise_ensemble"
V31_CONFIG_PATH = f"{V31_DIR}/stage2_best_val_config.json"

V32_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
V32_T4_PROB_PATH = f"{V32_DIR}/stage3_t4_specialist_probs.pkl"
V32_CONFIG_PATH = f"{V32_DIR}/stage3_t4_best_config.json"

V33_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
V33_VIEW_PROB_PATH = f"{V33_DIR}/stage3_3view_probs.pkl"
V33_CONFIG_PATH = f"{V33_DIR}/stage3_tta_best_config.json"

OUT_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
os.makedirs(OUT_DIR, exist_ok=True)

TASKS = [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
]

LABELS = {
    "promise_status": ["No", "Yes"],
    "verification_timeline": ["N/A", "already", "within_2_years", "between_2_and_5_years", "more_than_5_years"],
    "evidence_status": ["N/A", "No", "Yes"],
    "evidence_quality": ["N/A", "Clear", "Not Clear", "Misleading"],
}

T4_LABELS = ["Clear", "Not Clear", "Misleading"]

LABEL2ID = {t: {lab: i for i, lab in enumerate(labs)} for t, labs in LABELS.items()}
ID2LABEL = {t: {i: lab for i, lab in enumerate(labs)} for t, labs in LABELS.items()}

T4_LABEL2ID = {lab: i for i, lab in enumerate(T4_LABELS)}
T4_ID2LABEL = {i: lab for lab, i in T4_LABEL2ID.items()}

print("OUT_DIR:", OUT_DIR)
for p in [V31_CONFIG_PATH, V32_T4_PROB_PATH, V32_CONFIG_PATH, V33_VIEW_PROB_PATH, V33_CONFIG_PATH]:
    print(os.path.exists(p), p)


## 2. 讀取資料與快取

In [ ]:

def load_json_df(path):
    with open(path, "r", encoding="utf-8") as f:
        return pd.DataFrame(json.load(f))

train_df = load_json_df(TRAIN_PATH)
val_df = load_json_df(VAL_PATH)
test_df = load_json_df(TEST_PATH)

for df in [train_df, val_df]:
    for t in TASKS:
        df[t] = df[t].fillna("N/A").astype(str).replace({"": "N/A", "nan": "N/A"})

with open(V31_CONFIG_PATH, "r", encoding="utf-8") as f:
    v31_obj = json.load(f)
V31_BEST_CONFIG = v31_obj["best_config"]

with open(V33_CONFIG_PATH, "r", encoding="utf-8") as f:
    v33_obj = json.load(f)
V33_BEST_CONFIG = v33_obj["best_config"]

with open(V32_CONFIG_PATH, "r", encoding="utf-8") as f:
    V32_BEST_CONFIG_OBJ = json.load(f)

# stage3_t4_best_config.json 的格式是：
# {
#   "base": "stage2_best_val",
#   "mode": "blend",
#   "alpha": 0.05,
#   "spec_multipliers": {...},
#   ...
# }
V32_T4_ALPHA = float(V32_BEST_CONFIG_OBJ.get("alpha", 0.05))
V32_T4_SPEC_MULT = V32_BEST_CONFIG_OBJ.get("spec_multipliers", {"Clear": 1.0, "Not Clear": 1.0, "Misleading": 1.0})

with open(V33_VIEW_PROB_PATH, "rb") as f:
    view_obj = pickle.load(f)

with open(V32_T4_PROB_PATH, "rb") as f:
    t4_obj = pickle.load(f)

t4_val_probs = t4_obj["val_probs"]
t4_test_probs = t4_obj["test_probs"]

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

print("\nStage2_BEST_CONFIG:")
print(json.dumps(V31_BEST_CONFIG, ensure_ascii=False, indent=2))

print("\nStage3B_BEST_CONFIG:")
print(json.dumps(V33_BEST_CONFIG, ensure_ascii=False, indent=2))

print("\nStage3A T4 alpha:", V32_T4_ALPHA)
print("Stage3A T4 spec multipliers:", V32_T4_SPEC_MULT)

print("\nT4 probs:", t4_val_probs.shape, t4_test_probs.shape)


## 3. 評分與後處理工具

In [ ]:

def apply_hierarchy_to_pred_df(pred_df, force_quality_non_na_when_evidence_yes=False, probs=None):
    pred_df = pred_df.copy()

    no_mask = pred_df["promise_status"] == "No"
    pred_df.loc[no_mask, "verification_timeline"] = "N/A"
    pred_df.loc[no_mask, "evidence_status"] = "N/A"
    pred_df.loc[no_mask, "evidence_quality"] = "N/A"

    not_evidence_yes = pred_df["evidence_status"] != "Yes"
    pred_df.loc[not_evidence_yes, "evidence_quality"] = "N/A"

    if force_quality_non_na_when_evidence_yes and probs is not None:
        mask = (pred_df["evidence_status"] == "Yes") & (pred_df["evidence_quality"] == "N/A")
        if mask.any():
            q_probs = probs["evidence_quality"][mask.values].copy()
            q_probs[:, LABEL2ID["evidence_quality"]["N/A"]] = -1
            ids = q_probs.argmax(axis=1)
            pred_df.loc[mask, "evidence_quality"] = [ID2LABEL["evidence_quality"][int(i)] for i in ids]

    return pred_df

def f1_yes(y_true, y_pred):
    y_true_bin = (pd.Series(y_true).astype(str).values == "Yes").astype(int)
    y_pred_bin = (pd.Series(y_pred).astype(str).values == "Yes").astype(int)
    return f1_score(y_true_bin, y_pred_bin, pos_label=1, average="binary", zero_division=0)

def macro_f1_task(y_true, y_pred, task):
    return f1_score(
        y_true,
        y_pred,
        labels=LABELS[task],
        average="macro",
        zero_division=0,
    )

def calc_metrics(y_true_df, pred_df):
    out = {}
    out["promise_status_f1_yes"] = f1_yes(y_true_df["promise_status"], pred_df["promise_status"])
    out["verification_timeline_macro_f1"] = macro_f1_task(y_true_df["verification_timeline"], pred_df["verification_timeline"], "verification_timeline")
    out["evidence_status_f1_yes"] = f1_yes(y_true_df["evidence_status"], pred_df["evidence_status"])
    out["evidence_quality_macro_f1"] = macro_f1_task(y_true_df["evidence_quality"], pred_df["evidence_quality"], "evidence_quality")
    out["weighted_score"] = (
        0.20 * out["promise_status_f1_yes"]
        + 0.15 * out["verification_timeline_macro_f1"]
        + 0.30 * out["evidence_status_f1_yes"]
        + 0.35 * out["evidence_quality_macro_f1"]
    )
    return out

def print_metrics(m, title="metrics"):
    print("\n" + "="*80)
    print(title)
    print("="*80)
    for k, v in m.items():
        print(f"{k:36s}: {v:.6f}")

def adjust_probs(probs, multipliers):
    out = {}
    for t in TASKS:
        p = probs[t].copy()
        if t in multipliers:
            mult = np.array([multipliers[t].get(lab, 1.0) for lab in LABELS[t]], dtype=np.float32)
            p = p * mult.reshape(1, -1)
            p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
        out[t] = p
    return out

def probs_to_pred_df(df, probs, config=None):
    if config is None:
        config = {}

    multipliers = config.get("multipliers", {})
    force_q_non_na = config.get("force_quality_non_na_when_evidence_yes", False)
    t1_threshold = config.get("t1_threshold", None)
    t3_threshold = config.get("t3_threshold", None)

    probs2 = adjust_probs(probs, multipliers)

    pred = pd.DataFrame()
    pred["id"] = df["id"].values

    if t1_threshold is not None:
        yes_id = LABEL2ID["promise_status"]["Yes"]
        pred["promise_status"] = np.where(probs2["promise_status"][:, yes_id] >= t1_threshold, "Yes", "No")
    else:
        ids = probs2["promise_status"].argmax(axis=1)
        pred["promise_status"] = [ID2LABEL["promise_status"][int(i)] for i in ids]

    ids = probs2["verification_timeline"].argmax(axis=1)
    pred["verification_timeline"] = [ID2LABEL["verification_timeline"][int(i)] for i in ids]

    if t3_threshold is not None:
        yes_id = LABEL2ID["evidence_status"]["Yes"]
        pred_es = []
        for row_probs in probs2["evidence_status"]:
            if row_probs[yes_id] >= t3_threshold:
                pred_es.append("Yes")
            else:
                na_id = LABEL2ID["evidence_status"]["N/A"]
                no_id = LABEL2ID["evidence_status"]["No"]
                pred_es.append("N/A" if row_probs[na_id] >= row_probs[no_id] else "No")
        pred["evidence_status"] = pred_es
    else:
        ids = probs2["evidence_status"].argmax(axis=1)
        pred["evidence_status"] = [ID2LABEL["evidence_status"][int(i)] for i in ids]

    ids = probs2["evidence_quality"].argmax(axis=1)
    pred["evidence_quality"] = [ID2LABEL["evidence_quality"][int(i)] for i in ids]

    pred = apply_hierarchy_to_pred_df(
        pred,
        force_quality_non_na_when_evidence_yes=force_q_non_na,
        probs=probs2,
    )
    return pred


## 4. 3-view TTA 機率組合工具

In [ ]:

VIEW_PRESETS = {
    "head": {"head": 1.0, "middle": 0.0, "tail": 0.0},
    "middle": {"head": 0.0, "middle": 1.0, "tail": 0.0},
    "tail": {"head": 0.0, "middle": 0.0, "tail": 1.0},

    "head_mid": {"head": 0.70, "middle": 0.30, "tail": 0.0},
    "head_tail": {"head": 0.70, "middle": 0.0, "tail": 0.30},
    "mid_tail": {"head": 0.0, "middle": 0.50, "tail": 0.50},

    "even": {"head": 0.34, "middle": 0.33, "tail": 0.33},
    "head_mid_tail": {"head": 0.50, "middle": 0.25, "tail": 0.25},
    "mid_more": {"head": 0.30, "middle": 0.50, "tail": 0.20},
    "tail_more": {"head": 0.30, "middle": 0.20, "tail": 0.50},
    "head_tail_more": {"head": 0.50, "middle": 0.10, "tail": 0.40},
    "head_mid_more": {"head": 0.50, "middle": 0.40, "tail": 0.10},
}

def mix_views(stem_view_probs, split, task, preset_name):
    preset = VIEW_PRESETS[preset_name]
    out = None
    for view, w in preset.items():
        p = stem_view_probs[split][view][task]
        out = w * p if out is None else out + w * p
    return out

def build_tta_probs(split, config):
    weights = config.get("weights", {t: 0.5 for t in TASKS})
    view_presets = config.get("view_presets", {t: "head" for t in TASKS})

    out = {}
    for t in TASKS:
        preset_name = view_presets.get(t, "head")
        mac_p = mix_views(view_obj["mac"], split, t, preset_name)
        ckip_p = mix_views(view_obj["ckip"], split, t, preset_name)

        w_mac = weights.get(t, 0.5)
        p = w_mac * mac_p + (1 - w_mac) * ckip_p
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
        out[t] = p.astype(np.float32)

    return out

def eval_config_as_pred(df, split, config):
    probs = build_tta_probs(split, config)
    post_cfg = {k: v for k, v in config.items() if k not in ["weights", "view_presets"]}
    pred = probs_to_pred_df(df, probs, post_cfg)
    return pred, probs


## 5. T4 specialist blend 工具

In [ ]:

def t4_adjust_probs(spec_probs, spec_multipliers=None):
    p = spec_probs.copy()
    if spec_multipliers:
        mult = np.array([spec_multipliers.get(lab, 1.0) for lab in T4_LABELS], dtype=np.float32)
        p = p * mult.reshape(1, -1)
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
    return p

def get_base_quality_non_na_probs(base_probs, base_config):
    # 取得 base adjusted quality probs，轉成 Clear / Not Clear / Misleading 三類
    multipliers = base_config.get("multipliers", {})
    adjusted = adjust_probs(base_probs, multipliers)
    q = adjusted["evidence_quality"]

    non_na_ids = [
        LABEL2ID["evidence_quality"]["Clear"],
        LABEL2ID["evidence_quality"]["Not Clear"],
        LABEL2ID["evidence_quality"]["Misleading"],
    ]

    q3 = q[:, non_na_ids].copy()
    q3 = q3 / np.clip(q3.sum(axis=1, keepdims=True), 1e-12, None)
    return q3

def compose_with_t4_blend(
    df,
    base_pred,
    base_probs,
    base_config,
    spec_probs,
    alpha=0.05,
    spec_multipliers=None,
):
    out = base_pred.copy()

    mask = out["evidence_status"] == "Yes"

    base_q3 = get_base_quality_non_na_probs(base_probs, base_config)
    spec_q3 = t4_adjust_probs(spec_probs, spec_multipliers=spec_multipliers)

    blend = (1 - alpha) * base_q3 + alpha * spec_q3
    blend = blend / np.clip(blend.sum(axis=1, keepdims=True), 1e-12, None)

    ids = blend.argmax(axis=1)
    labels = np.array([T4_ID2LABEL[int(i)] for i in ids])

    out.loc[mask, "evidence_quality"] = labels[mask.values]
    out.loc[~mask, "evidence_quality"] = "N/A"
    out = apply_hierarchy_to_pred_df(out)

    return out


## 6. 重建 Stage2 / Stage3A / Stage3B / Stage3C 分數

In [ ]:

def copy_config(cfg):
    return json.loads(json.dumps(cfg))

# Stage2-like：全部 head view + Stage2 config
v31_like_config = copy_config(V31_BEST_CONFIG)
v31_like_config["view_presets"] = {t: "head" for t in TASKS}

# Stage3B：使用 Stage3B best config
v33_config = copy_config(V33_BEST_CONFIG)

# Stage3C：先使用 Stage3B best config，再把 T4 改成 Stage3A 的 T4 blend
v34_base_config = copy_config(V33_BEST_CONFIG)

v31_pred, v31_probs = eval_config_as_pred(val_df, "val", v31_like_config)
v31_metrics = calc_metrics(val_df, v31_pred)
print_metrics(v31_metrics, "v31 reconstructed")

# Stage3A reconstructed：Stage2 base + T4 blend
v32_pred = compose_with_t4_blend(
    df=val_df,
    base_pred=v31_pred,
    base_probs=v31_probs,
    base_config=v31_like_config,
    spec_probs=t4_val_probs,
    alpha=V32_T4_ALPHA,
    spec_multipliers=V32_T4_SPEC_MULT,
)
v32_metrics = calc_metrics(val_df, v32_pred)
print_metrics(v32_metrics, "v32_00 reconstructed: v31 + T4 blend")

v33_pred, v33_probs = eval_config_as_pred(val_df, "val", v33_config)
v33_metrics = calc_metrics(val_df, v33_pred)
print_metrics(v33_metrics, "v33 reconstructed")

# Stage3C direct：Stage3B base + T4 blend
stage3_direct_pred = compose_with_t4_blend(
    df=val_df,
    base_pred=v33_pred,
    base_probs=v33_probs,
    base_config=v33_config,
    spec_probs=t4_val_probs,
    alpha=V32_T4_ALPHA,
    spec_multipliers=V32_T4_SPEC_MULT,
)
stage3_direct_metrics = calc_metrics(val_df, stage3_direct_pred)
print_metrics(stage3_direct_metrics, "v34 direct: v33 timeline + v32 T4 blend")

for name, pred in [
    ("v31", v31_pred),
    ("v32", v32_pred),
    ("Stage3B", v33_pred),
    ("v33", stage3_direct_pred),
]:
    print("\n", name)
    for t in TASKS:
        print(t, pred[t].value_counts().to_dict())


## 7. v34 輕量搜尋：微調 T4 alpha、T4 multipliers、T2 view

In [ ]:

def eval_v34_combo(config, alpha, spec_multipliers):
    base_pred, base_probs = eval_config_as_pred(val_df, "val", config)
    pred = compose_with_t4_blend(
        df=val_df,
        base_pred=base_pred,
        base_probs=base_probs,
        base_config=config,
        spec_probs=t4_val_probs,
        alpha=alpha,
        spec_multipliers=spec_multipliers,
    )
    metrics = calc_metrics(val_df, pred)
    return metrics["weighted_score"], metrics, pred

# 以 Stage3B config 為主，只做小範圍搜尋，避免過度 fitting
alpha_values = [0.0, 0.02, 0.03, 0.05, 0.07, 0.10, 0.15]
view_values_t2 = ["head", "tail", "head_tail", "tail_more", "head_mid_tail", "mid_more"]
t4_mult_candidates = [
    V32_T4_SPEC_MULT,
    {"Clear": 0.5, "Not Clear": 0.25, "Misleading": 0.5},
    {"Clear": 0.75, "Not Clear": 0.50, "Misleading": 0.5},
    {"Clear": 1.0, "Not Clear": 0.50, "Misleading": 0.5},
    {"Clear": 1.0, "Not Clear": 0.75, "Misleading": 0.5},
    {"Clear": 1.0, "Not Clear": 1.0, "Misleading": 0.5},
    {"Clear": 0.5, "Not Clear": 0.5, "Misleading": 0.5},
]

best = None
rows = []

for t2_view in view_values_t2:
    cfg = copy_config(V33_BEST_CONFIG)
    cfg.setdefault("view_presets", {})
    cfg["view_presets"]["verification_timeline"] = t2_view

    for alpha in alpha_values:
        for spec_mult in t4_mult_candidates:
            score, metrics, pred = eval_v34_combo(cfg, alpha, spec_mult)

            row = {
                "t2_view": t2_view,
                "alpha": alpha,
                "spec_mult": json.dumps(spec_mult, ensure_ascii=False),
                **metrics,
                "pred_timeline_already": int((pred["verification_timeline"] == "already").sum()),
                "pred_timeline_within_2": int((pred["verification_timeline"] == "within_2_years").sum()),
                "pred_timeline_2_5": int((pred["verification_timeline"] == "between_2_and_5_years").sum()),
                "pred_timeline_more_5": int((pred["verification_timeline"] == "more_than_5_years").sum()),
                "pred_quality_Clear": int((pred["evidence_quality"] == "Clear").sum()),
                "pred_quality_Not_Clear": int((pred["evidence_quality"] == "Not Clear").sum()),
                "pred_quality_NA": int((pred["evidence_quality"] == "N/A").sum()),
                "pred_quality_Misleading": int((pred["evidence_quality"] == "Misleading").sum()),
            }
            rows.append(row)

            if best is None or score > best[0]:
                best = (score, metrics, pred, cfg, alpha, spec_mult)

search_df = pd.DataFrame(rows).sort_values("weighted_score", ascending=False)
search_df.to_csv(f"{OUT_DIR}/stage3_blend_search.csv", index=False, encoding="utf-8-sig")

best_score, best_metrics, best_pred, best_base_config, best_alpha, best_spec_mult = best

print_metrics(best_metrics, "BEST Stage3C val1000")
print("best_alpha:", best_alpha)
print("best_spec_mult:", best_spec_mult)
print("best_base_config:")
print(json.dumps(best_base_config, ensure_ascii=False, indent=2))

for t in TASKS:
    print("\n", t)
    print(best_pred[t].value_counts().to_dict())

display(search_df.head(20))


## 8. 儲存 val1000 錯誤分析與 config

In [ ]:

def save_val_outputs(pred, metrics, tag):
    val_full = pd.concat(
        [
            val_df[["id", "data", "company", "page_number"]].reset_index(drop=True),
            val_df[TASKS].add_prefix("true_").reset_index(drop=True),
            pred[TASKS].add_prefix("pred_").reset_index(drop=True),
        ],
        axis=1,
    )

    for t in TASKS:
        val_full[f"err_{t}"] = val_full[f"true_{t}"] != val_full[f"pred_{t}"]

    val_full["any_error"] = val_full[[f"err_{t}" for t in TASKS]].any(axis=1)

    val_full.to_csv(f"{OUT_DIR}/{tag}_val1000_predictions.csv", index=False, encoding="utf-8-sig")
    val_full.to_excel(f"{OUT_DIR}/{tag}_val1000_error_analysis.xlsx", index=False)

    with pd.ExcelWriter(f"{OUT_DIR}/{tag}_val1000_confusion.xlsx", engine="openpyxl") as writer:
        for t in TASKS:
            cm = confusion_matrix(val_df[t], pred[t], labels=LABELS[t])
            cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in LABELS[t]], columns=[f"pred_{x}" for x in LABELS[t]])
            cm_df.to_excel(writer, sheet_name=t[:31])

    return val_full

best_val_full = save_val_outputs(best_pred, best_metrics, "stage3_best_val")
direct_val_full = save_val_outputs(stage3_direct_pred, stage3_direct_metrics, "stage3_direct")

v34_config_obj = {
    "base_config": best_base_config,
    "t4_alpha": best_alpha,
    "t4_spec_multipliers": best_spec_mult,
    "v31_metrics": v31_metrics,
    "v32_reconstructed_metrics": v32_metrics,
    "v33_metrics": v33_metrics,
    "stage3_direct_metrics": stage3_direct_metrics,
    "best_metrics": best_metrics,
}

with open(f"{OUT_DIR}/stage3_best_config.json", "w", encoding="utf-8") as f:
    json.dump(v34_config_obj, f, ensure_ascii=False, indent=2)

print("saved config and val outputs to:", OUT_DIR)
display(best_val_full["any_error"].value_counts())
display(best_val_full.head())


## 9. 產生 test2000 submissions

In [ ]:

def compact_text(s):
    return re.sub(r"\s+", "", str(s))

train_map = {}
for _, row in train_df.iterrows():
    train_map[compact_text(row["data"])] = {t: row[t] for t in TASKS}

def build_v34_test_pred(base_config, alpha, spec_multipliers):
    base_pred, base_probs = eval_config_as_pred(test_df, "test", base_config)

    pred = compose_with_t4_blend(
        df=test_df,
        base_pred=base_pred,
        base_probs=base_probs,
        base_config=base_config,
        spec_probs=t4_test_probs,
        alpha=alpha,
        spec_multipliers=spec_multipliers,
    )

    # exact-match override
    override_count = 0
    for i, row in test_df.iterrows():
        key = compact_text(row["data"])
        if key in train_map:
            for t in TASKS:
                pred.loc[i, t] = train_map[key][t]
            override_count += 1

    pred = apply_hierarchy_to_pred_df(pred)
    return pred, override_count

def make_test_submission(base_config, alpha, spec_multipliers, tag):
    pred, override_count = build_v34_test_pred(base_config, alpha, spec_multipliers)

    sub = pred[["id"] + TASKS].copy()
    path = f"{OUT_DIR}/{tag}_test2000_submission.csv"
    sub.to_csv(path, index=False, encoding="utf-8-sig")

    print("\n", tag, "saved:", path)
    print("override_count:", override_count)
    for t in TASKS:
        print(t, sub[t].value_counts().to_dict())

    return sub, path

# best：val1000 搜尋最佳
best_sub, best_path = make_test_submission(
    best_base_config,
    best_alpha,
    best_spec_mult,
    "stage3_best_val",
)

# direct：固定 v33 + v32_00 T4 blend，不使用 light search 過度調整
direct_sub, direct_path = make_test_submission(
    v34_base_config,
    V32_T4_ALPHA,
    V32_T4_SPEC_MULT,
    "stage3_direct",
)

# conservative：提高 T1/T3 threshold，並降低 T4 specialist 影響
conservative_cfg = copy_config(best_base_config)
if conservative_cfg.get("t1_threshold") is not None:
    conservative_cfg["t1_threshold"] = min(float(conservative_cfg["t1_threshold"]) + 0.04, 0.80)
if conservative_cfg.get("t3_threshold") is not None:
    conservative_cfg["t3_threshold"] = min(float(conservative_cfg["t3_threshold"]) + 0.04, 0.80)

conservative_alpha = min(best_alpha, 0.05)
conservative_sub, conservative_path = make_test_submission(
    conservative_cfg,
    conservative_alpha,
    best_spec_mult,
    "stage3_conservative",
)

# balanced：保留 v33 timeline，但避免 within_2_years / Not Clear 太激進
balanced_cfg = copy_config(best_base_config)
balanced_cfg.setdefault("multipliers", {})

if "verification_timeline" in balanced_cfg["multipliers"]:
    if "within_2_years" in balanced_cfg["multipliers"]["verification_timeline"]:
        balanced_cfg["multipliers"]["verification_timeline"]["within_2_years"] = max(
            1.0,
            float(balanced_cfg["multipliers"]["verification_timeline"]["within_2_years"]) * 0.75
        )

balanced_spec_mult = copy_config(best_spec_mult)
if "Not Clear" in balanced_spec_mult:
    balanced_spec_mult["Not Clear"] = max(0.25, float(balanced_spec_mult["Not Clear"]) * 0.85)

balanced_sub, balanced_path = make_test_submission(
    balanced_cfg,
    best_alpha,
    balanced_spec_mult,
    "stage3_balanced",
)

print("\nSubmission paths:")
print(best_path)
print(direct_path)
print(conservative_path)
print(balanced_path)


## 10. 報告摘要

In [ ]:

report = []
report.append("# Stage3C combine Stage3B timeline + Stage3A T4 report")
report.append("")
report.append("## Metrics")
report.append("### Stage2 reconstructed")
for k, v in v31_metrics.items():
    report.append(f"- {k}: {v:.6f}")
report.append("### Stage3A reconstructed")
for k, v in v32_metrics.items():
    report.append(f"- {k}: {v:.6f}")
report.append("### Stage3B reconstructed")
for k, v in v33_metrics.items():
    report.append(f"- {k}: {v:.6f}")
report.append("### Stage3C direct")
for k, v in stage3_direct_metrics.items():
    report.append(f"- {k}: {v:.6f}")
report.append("### Stage3C best")
for k, v in best_metrics.items():
    report.append(f"- {k}: {v:.6f}")

report.append("")
report.append("## Best config")
report.append("```json")
report.append(json.dumps(v34_config_obj, ensure_ascii=False, indent=2))
report.append("```")

report.append("")
report.append("## Test distribution: stage3_best_val")
for t in TASKS:
    report.append(f"### {t}")
    for lab, cnt in best_sub[t].value_counts().to_dict().items():
        report.append(f"- {lab}: {cnt}")

report_md = "\n".join(report)

with open(f"{OUT_DIR}/stage3_report.md", "w", encoding="utf-8") as f:
    f.write(report_md)

print(report_md[:5000])
print("saved:", f"{OUT_DIR}/stage3_report.md")
